# 02.5 Transfer Learning

The core idea of transfer learning is:

- do not learn everything from scratch
- reuse the feature-extraction ability of an existing model

Key concepts:

- pretraining
- backbone
- classification head
- freeze parameters
- unfreeze parameters
- fine-tuning

## Learning Goals

After this notebook, you should be able to:

1. Understand the basic transfer-learning workflow.
2. Distinguish the backbone from the head.
3. Freeze and unfreeze parameters.
4. Replace a classification head.
5. Understand the difference between head-only training and fine-tuning.
6. Run a minimal transfer-learning workflow in an offline environment.

## Offline Note

To guarantee this notebook runs in the current offline environment, we use:

- `models.resnet18(weights=None)`

This means it will not benefit from pretrained weights, but it can still demonstrate the full transfer-learning workflow.

If you later work in an environment with cached weights or internet access, you can change it to:

- `models.resnet18(weights=models.ResNet18_Weights.DEFAULT)`

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

## Prepare a Dataset That Can Be Fed into ResNet

`digits` is grayscale `(1, 8, 8)`, while a standard `ResNet` usually expects RGB images `(3, H, W)`.

So we do three transformations:

1. scale to `0~1`
2. repeat grayscale into 3 channels
3. resize to a size more suitable for `ResNet`

In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

X_train_full, X_val, y_train_full, y_val = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

# To keep the notebook fast, we use a smaller subset for demonstration.
X_train = X_train_full[:256]
y_train = y_train_full[:256]
X_val_small = X_val[:96]
y_val_small = y_val[:96]

print("train subset shape =", X_train.shape)
print("val subset shape =", X_val_small.shape)

In [ ]:
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Resize((64, 64), antialias=True),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class DigitsRGBDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(self.labels[index]), dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_ds = DigitsRGBDataset(X_train, y_train, transform=transform)
val_ds = DigitsRGBDataset(X_val_small, y_val_small, transform=transform)

xb, yb = train_ds[0]
print("one image shape =", xb.shape)
print("one label =", yb)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch image shape =", xb.shape)
print("batch label shape =", yb.shape)

## Build the Backbone + Head

We use `resnet18` here as the backbone.

The classification head is the final `fc` layer.


In [ ]:
model = models.resnet18(weights=None)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

print(model.fc)
print("fc in_features =", in_features)

## Freeze the Backbone and Train Only the Head

Head-only training means:

- backbone / backbone frozen
- only the final classification head is updated

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print("trainable params =", trainable_params)
print("all params =", all_params)

## Training Helper Functions

To keep the notebook clean, we wrap the training logic into helper functions first.


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 5. Head-only Training

Because the backbone is frozen, the optimizer only receives parameters with `requires_grad=True`.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)

head_history = []
for epoch in range(1, 3):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    head_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "phase": "head_only",
    })
    print(f"head_only epoch={epoch} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

## Unfreeze the Last Part and Fine-Tune

Next we unfreeze only `layer4` and `fc`, so the later part of the model can update together.

This is a common partial fine-tuning setup.


In [ ]:
for param in model.layer4.parameters():
    param.requires_grad = True

trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params after unfreezing layer4 =", trainable_params_after)

In [ ]:
finetune_optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

finetune_history = []
for epoch in range(1, 3):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=finetune_optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    finetune_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "phase": "partial_finetune",
    })
    print(f"partial_finetune epoch={epoch} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

## Compare Head-only and Partial Fine-Tuning

The key here is not that one result must always be better, but that the two stages train in different ways.


In [ ]:
history_df = torch.tensor([])  # placeholder to keep notebook structure simple
import pandas as pd

comparison = pd.DataFrame(head_history + finetune_history)
print(comparison)

## Practical Notes on Transfer Learning

In real projects, transfer learning is usually more suitable when:

- the dataset is relatively small
- the new task is reasonably close to existing vision tasks
- pretrained weights are available

The goal of this notebook is to make sure you first understand the workflow end to end.


In [ ]:
# Exercise 1
# In one sentence, explain why we freeze the backbone during head-only training.


Exercise 1 Reference Answer

We freeze the backbone during head-only training to reuse pretrained features cheaply and avoid immediately distorting them with a small task-specific dataset.

In [ ]:
# Exercise 2
# If you want more layers to participate in fine-tuning, where would you start modifying this notebook?


Exercise 2 Reference Answer

Start by unfreezing the later backbone blocks first, because they contain more task-specific visual features than the earliest edge and texture layers.

## Summary

The most important outcome of this notebook is understanding how transfer learning is organized, not just knowing the name of a model.

You should now be able to answer:

1. What are the backbone and the head?
2. Why do we sometimes freeze first and unfreeze later?
3. What is the difference between head-only training and partial fine-tuning?
4. Why does this notebook run offline but still not fully benefit from pretraining?

Suggested next step:

- Move to the Phase 2 project notebook and compare different image models through experiments.